### *Jorge Edgardo Viera*  


## Módulo 4 · Tarea 5 — Multiprocessing y concurrencia asíncrona en Python

**Nota:** *Para esta tarea de paralelismo, lo más sano fue trabajar con archivos `.py` y ejecutarlos desde la terminal, no desde el notebook. Es más fiel a cómo se usa multiprocessing en la vida real y evitamos posibles errores.*


### ***1) Multiprocessing básico: procesos paralelos***

In [1]:
import multiprocessing
import time

numeros = [1, 2, 3, 4, 5]

# ---------- VERSIÓN SECUENCIAL ----------
"""Con un bucle for, calculamos el cuadrado de cada número en la lista y lo almacenamos en otra lista.
Esta versión es secuencial, es decir, cada cálculo se realiza uno tras otro."""
def calcular_cuadrados_secuencial(lista):
    resultados = []
    for n in lista:
        resultados.append(n ** 2)
    return resultados

inicio = time.perf_counter() # Usamos time.perf_counter() para medir el tiempo de ejecución que es más preciso que time.time()
resultados_secuencial = calcular_cuadrados_secuencial(numeros)
fin = time.perf_counter()
tiempo_secuencial = fin - inicio

print("Resultados (secuencial):", resultados_secuencial)
print(f"Tiempo secuencial: {tiempo_secuencial:.6f} segundos\n")


# ---------- VERSIÓN PARALELA ----------
def calcular_cuadrado_proceso(n, queue):
    resultado = n ** 2
    queue.put((n, resultado))  # metemos el resultado en la cola compartida

if __name__ == "__main__":
    queue = multiprocessing.Queue()
    procesos = []

    inicio = time.perf_counter()

    # Creamos un proceso por cada número
    for n in numeros:
        p = multiprocessing.Process(target=calcular_cuadrado_proceso, args=(n, queue)) # Uso de queue para pasar resultados entre procesos
        procesos.append(p)
        p.start()  # lanza el proceso

    # Esperamos a que todos terminen
    for p in procesos:
        p.join()

    fin = time.perf_counter()
    tiempo_paralelo = fin - inicio

    # Recogemos los resultados de la cola
    resultados_paralelo = []
    while not queue.empty():
        resultados_paralelo.append(queue.get())

    # Ordenamos los resultados por el número original
    resultados_paralelo.sort(key=lambda x: x[0])
    resultados_paralelo = [resultado for _, resultado in resultados_paralelo]

    print("Resultados (paralelo):", resultados_paralelo)
    print(f"Tiempo paralelo: {tiempo_paralelo:.6f} segundos\n")

    # ---------- COMPARACIÓN ----------
    print(f"Tiempo secuencial: {tiempo_secuencial:.6f} s")
    print(f"Tiempo paralelo:   {tiempo_paralelo:.6f} s")

Resultados (secuencial): [1, 4, 9, 16, 25]
Tiempo secuencial: 0.000162 segundos

Resultados (paralelo): []
Tiempo paralelo: 0.503407 segundos

Tiempo secuencial: 0.000162 s
Tiempo paralelo:   0.503407 s


### ***2) Pool de procesos***

**Reminder**: *EJECUTAR ESTE CODIGO EN UN SCRIPT `.PY`*

In [ ]:
import multiprocessing
import time
import math
"""Función para calcular el factorial de un número. Esta función será utilizada por los procesos en el pool."""
def calcular_factorial(n):
    return math.factorial(n)

if __name__ == "__main__": # Protección para evitar que el código se ejecute al importar el módulo en Windows
    numeros = list(range(1, 11))  # del 1 al 10

    tamanos_pool = [2, 4, 8] # Pruebas con diferentes tamaños de pool,para buenas practicamos no usamos la letra "n".

    for tamano in tamanos_pool: # Iteramos sobre los diferentes tamaños de pool
        inicio = time.perf_counter()

        with multiprocessing.Pool(processes=tamano) as pool: # Uso de multiprocessing.Pool para crear un pool de procesos
            resultados = pool.map(calcular_factorial, numeros)

        fin = time.perf_counter() # Medimos el tiempo de ejecución del pool de procesos
        tiempo = fin - inicio

        print(f"Pool con {tamano} procesos:")
        print(f"  Resultados: {resultados}")
        print(f"  Tiempo: {tiempo:.6f} segundos\n")

### ***3) Introducción a async/await (simulación de E/S)***

**Observamos que los tiempos de ejecucion son...**
- Secuencial: ~5 segundos (5 tareas × 1 segundo, una detrás de otra).
- Concurrente: ~1 segundo (las 5 tareas "esperan" a la vez, así que el tiempo total es aproximadamente el de la tarea más larga, no la suma de todas).

**Aqui la clave conceptual de lo que sucede**

- *Concurrencia* = dar la impresión de que varias cosas pasan "a la vez", aunque en realidad solo hay un hilo que va saltando de tarea en tarea cada vez que una se queda esperando (como un camarero que atiende varias mesas, yendo de una a otra mientras cada mesa "piensa" qué pedir).

- *Paralelismo* = varias cosas pasan literalmente al mismo tiempo, en núcleos de CPU distintos (como tener varios camareros, cada uno atendiendo su propia mesa simultáneamente).



In [ ]:
import asyncio # Importamos asyncio para manejar la ejecución asíncrona de tareas
import time

"""Funcion asíncrona que simula una tarea que tarda 1 segundo en completarse. 
Esta función será utilizada para demostrar la diferencia entre ejecución secuencial y concurrente."""
async def tarea(i):
    print(f"Tarea {i} iniciada")
    await asyncio.sleep(1)  # simula una espera de E/S (ej: leer disco, red)
    print(f"Tarea {i} terminada")
    return i

# ---------- VERSIÓN SECUENCIAL (await uno por uno) ----------
async def ejecutar_secuencial():
    resultados = []
    for i in range(5):
        resultado = await tarea(i)  # espera a que termine ANTES de seguir
        resultados.append(resultado)
    return resultados

# ---------- VERSIÓN CONCURRENTE (asyncio.gather) ----------
async def ejecutar_concurrente():
    tareas = [tarea(i) for i in range(5)]
    resultados = await asyncio.gather(*tareas)  # lanza todas "a la vez"
    return resultados

"""Esta función principal ejecuta ambas versiones (secuencial y concurrente)
 y mide el tiempo que tarda cada una en completarse."""
async def main():
    print("=== Ejecución secuencial ===")
    inicio = time.perf_counter()
    resultados_seq = await ejecutar_secuencial()
    fin = time.perf_counter()
    tiempo_secuencial = fin - inicio
    print(f"Resultados: {resultados_seq}")
    print(f"Tiempo secuencial: {tiempo_secuencial:.2f} segundos\n")

    print("=== Ejecución concurrente ===")
    inicio = time.perf_counter()
    resultados_con = await ejecutar_concurrente()
    fin = time.perf_counter()
    tiempo_concurrente = fin - inicio
    print(f"Resultados: {resultados_con}")
    print(f"Tiempo concurrente: {tiempo_concurrente:.2f} segundos\n")

    print(f"Secuencial:  {tiempo_secuencial:.2f} s")
    print(f"Concurrente: {tiempo_concurrente:.2f} s")

# Uso de await main() para ejecutar la función principal de manera asíncrona. Esto permite que el código se ejecute correctamente en un entorno que soporte asyncio.
await main()

=== Ejecución secuencial ===
Tarea 0 iniciada
Tarea 0 terminada
Tarea 1 iniciada
Tarea 1 terminada
Tarea 2 iniciada
Tarea 2 terminada
Tarea 3 iniciada
Tarea 3 terminada
Tarea 4 iniciada
Tarea 4 terminada
Resultados: [0, 1, 2, 3, 4]
Tiempo secuencial: 5.06 segundos

=== Ejecución concurrente ===
Tarea 0 iniciada
Tarea 1 iniciada
Tarea 2 iniciada
Tarea 3 iniciada
Tarea 4 iniciada
Tarea 0 terminada
Tarea 1 terminada
Tarea 2 terminada
Tarea 3 terminada
Tarea 4 terminada
Resultados: [0, 1, 2, 3, 4]
Tiempo concurrente: 1.00 segundos

Secuencial:  5.06 s
Concurrente: 1.00 s


### ***4) Procesamiento de archivos locales con async/await***

En este codigo se implementa `asyncio.to_thread` ya que no existe una versión `async def` nativa de leer archivos en el estándar de Python y no podemos llamar a `open()`porque bloquearíamos todo el event loop mientras se lee el archivo, y perderías toda la ventaja de la concurrencia. Entonces, `asyncio.to_thread(funcion, *args)`lo resuelve ejecutando la función bloqueante normal en un **thread separado del pool** sin bloquear el event loop principal.

*Qué es una ***coroutine***..., nombre que aprendi a aplicar en el uso de `asyncio`...

Una coroutine es una función especial que puede pausarse a sí misma y reanudarse más tarde, sin perder su estado (sus variables locales, en qué punto se quedó, etc.).

In [ ]:
import asyncio
import os
import time

CARPETA = "datos"

# ---------- GENERAR ARCHIVOS DE PRUEBA ----------
def generar_archivos():
    os.makedirs(CARPETA, exist_ok=True)

    contenidos = {
        "archivo1.txt": "Python es un lenguaje de programación.\nEs muy popular para ciencia de datos.\n",
        "archivo2.txt": "El asincronismo permite manejar muchas tareas de E/S a la vez.\nasyncio es la librería estándar para esto.\nFunciona con un solo hilo.\n",
        "archivo3.txt": "Hola mundo.\n",
        "archivo4.txt": "La concurrencia y el paralelismo son conceptos distintos.\nLa concurrencia no siempre implica varios núcleos.\nEl paralelismo sí usa varios núcleos físicos.\nEs importante no confundirlos.\n",
        "archivo5.txt": "Este archivo tiene pocas palabras.\n",
    }

    for nombre, texto in contenidos.items():
        ruta = os.path.join(CARPETA, nombre)
        if not os.path.exists(ruta):
            with open(ruta, "w", encoding="utf-8") as f: # Uso de encoding="utf-8" para soportar caracteres especiales
                f.write(texto)

    return list(contenidos.keys())


# ---------- LECTURA BLOQUEANTE (se ejecutará en un hilo) ----------
def leer_y_analizar(ruta):
    # Esto simula una operación bloqueante normal de E/S
    with open(ruta, "r", encoding="utf-8") as f: # Uso de encoding="utf-8" para soportar caracteres especiales
        contenido = f.read()

    lineas = contenido.splitlines()
    palabras = contenido.split()

    return {
        "archivo": os.path.basename(ruta),
        "lineas": len(lineas),
        "palabras": len(palabras),
    }


# ---------- COROUTINE QUE ENVUELVE LA LECTURA CON to_thread ----------
async def procesar_archivo(ruta):
    try:
        resultado = await asyncio.to_thread(leer_y_analizar, ruta) # to_thread permite ejecutar la función bloqueante en un hilo separado
        return resultado
    except FileNotFoundError:
        return {"archivo": os.path.basename(ruta), "error": "Archivo no encontrado"}
    except Exception as e:
        return {"archivo": os.path.basename(ruta), "error": str(e)}


async def main():
    nombres = generar_archivos()

    # Añadimos un archivo que NO existe a propósito, para probar el manejo de errores
    nombres.append("archivo_inexistente.txt")

    rutas = [os.path.join(CARPETA, nombre) for nombre in nombres]

    inicio = time.perf_counter()

    # Lanza la lectura de todos los archivos concurrentemente
    tareas = [procesar_archivo(ruta) for ruta in rutas]
    resultados = await asyncio.gather(*tareas)

    fin = time.perf_counter()

    # Separamos los resultados correctos de los que dieron error
    correctos = [r for r in resultados if "error" not in r]
    con_error = [r for r in resultados if "error" in r]

    # Ordenamos por número de palabras, descendente
    correctos.sort(key=lambda r: r["palabras"], reverse=True)

    print("=== Resumen de archivos (ordenado por nº de palabras, descendente) ===")
    print(f"{'Archivo':<25}{'Líneas':<10}{'Palabras':<10}")
    print("-" * 45)
    for r in correctos:
        print(f"{r['archivo']:<25}{r['lineas']:<10}{r['palabras']:<10}")

    if con_error:
        print("\n=== Archivos con error (no interrumpieron la ejecución) ===")
        for r in con_error:
            print(f"{r['archivo']:<25} -> {r['error']}")

    print(f"\nTiempo total: {fin - inicio:.4f} segundos")

# En notebook: await main()
# En script .py: asyncio.run(main())
await main()

=== Resumen de archivos (ordenado por nº de palabras, descendente) ===
Archivo                  Líneas    Palabras  
---------------------------------------------
archivo4.txt             4         26        
archivo2.txt             3         23        
archivo1.txt             2         13        
archivo5.txt             1         5         
archivo3.txt             1         2         

=== Archivos con error (no interrumpieron la ejecución) ===
archivo_inexistente.txt   -> Archivo no encontrado

Tiempo total: 0.0604 segundos


### ***5) Combinando multiprocessing y async/await***

**En este codigo combinamos los tres ejercicios anteriores en un pipeline real.**

**Aviso importante:**
*Como este ejercicio usa multiprocessing.Pool (igual que el ejercicio 2), debes ejecutarlo como archivo .py desde la terminal, no desde el notebook.*

In [ ]:
# Importamos los módulos necesarios para la generación de archivos CSV, procesamiento concurrente y manejo de errores.
#-----------DETALLE DE CADA MÓDULO IMPORTADO-----------
import multiprocessing # Permite crear procesos independientes para ejecutar tareas en paralelo, aprovechando múltiples núcleos de CPU.
import asyncio # Permite escribir código concurrente usando corutinas, facilitando la ejecución de tareas de E/S sin bloquear el hilo principal.
import csv # Proporciona funcionalidades para leer y escribir archivos CSV de manera sencilla y eficiente.
import os # Permite interactuar con el sistema operativo, como crear carpetas, manejar rutas de archivos y verificar la existencia de archivos.
import random # Permite generar números aleatorios y seleccionar elementos de manera aleatoria, útil para simular datos en los CSV.
import time    # Permite medir el tiempo de ejecución de bloques de código, útil para comparar rendimiento entre versiones secuenciales y concurrentes.
import statistics # Proporciona funciones para calcular estadísticas básicas como media, desviación estándar, etc., útil para analizar los datos generados en los CSV.
import logging # Permite registrar mensajes de error y eventos importantes en un archivo de log, facilitando la depuración y el seguimiento de errores sin interrumpir la ejecución del programa.

# ---------- CONFIGURACIÓN ----------
CARPETA_SALIDA = "salida" # Carpeta donde se guardarán los CSV generados
N_FICHEROS = 4 # Número de ficheros CSV a generar
FILAS_POR_FICHERO = 200_000 # Número de filas por fichero CSV
GRUPOS = ["A", "B", "C", "D"] # Grupos posibles para los datos

# Para gestionar errores sin interrumpir la ejecución, configuramos el logging para registrar errores 
# en un archivo llamado "errores.log" con un formato que incluye la fecha y hora.
logging.basicConfig(
    filename="errores.log",
    level=logging.ERROR,
    format="%(asctime)s - %(message)s",
) # Configuración del logging para registrar errores en un archivo llamado "errores.log" con un formato que incluye la fecha y hora.


# ================================================================
# a) Generación paralela de CSV con multiprocessing.Pool
# ================================================================
def generar_csv(i):
    """Crea un CSV con FILAS_POR_FICHERO filas. Se ejecuta en un proceso del Pool."""
    ruta = os.path.join(CARPETA_SALIDA, f"datos_{i}.csv")
    random.seed(os.getpid() + i)
    try:
        with open(ruta, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["id", "grupo", "valor"])
            for fila_id in range(FILAS_POR_FICHERO):
                grupo = random.choice(GRUPOS) # Uso de random.choice para seleccionar un grupo aleatorio de la lista GRUPOS
                valor = round(random.uniform(0, 100), 4)
                writer.writerow([fila_id, grupo, valor])
        return ruta
    except Exception as e:
        return f"ERROR::{ruta}::{e}"

"""Funcion para generar múltiples ficheros CSV en paralelo usando multiprocessing.Pool.
Crea la carpeta de salida si no existe y lanza un pool de procesos para generar los ficheros."""
def generar_ficheros_paralelo(n_ficheros, procesos):
    os.makedirs(CARPETA_SALIDA, exist_ok=True)
    with multiprocessing.Pool(processes=procesos) as pool:
        return pool.map(generar_csv, range(n_ficheros))

"""Funcion para generar múltiples ficheros CSV de manera secuencial. 
Crea la carpeta de salida si no existe y genera los ficheros uno por uno."""
def generar_ficheros_secuencial(n_ficheros):
    os.makedirs(CARPETA_SALIDA, exist_ok=True)
    return [generar_csv(i) for i in range(n_ficheros)]


# ================================================================
# b) Lectura y análisis concurrente con asyncio + to_thread
# ================================================================
def leer_y_analizar_csv(ruta):
    """Función bloqueante: lee el CSV y calcula sus métricas. Corre en un hilo (to_thread)."""
    n_filas = 0
    valores = []
    conteo_grupo = {g: 0 for g in GRUPOS}

    with open(ruta, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f) # Lee el CSV como un diccionario para acceder a las columnas por nombre
        for fila in reader:
            n_filas += 1
            valores.append(float(fila["valor"]))
            conteo_grupo[fila["grupo"]] = conteo_grupo.get(fila["grupo"], 0) + 1

    media = statistics.mean(valores) if valores else 0.0
    desviacion = statistics.stdev(valores) if len(valores) > 1 else 0.0

    return {
        "archivo": os.path.basename(ruta),
        "filas": n_filas,
        "media": media,
        "desviacion": desviacion,
        "conteo_grupo": conteo_grupo,
    }


async def procesar_csv(ruta, semaforo):
    # f) Semaphore: limita cuántas lecturas concurrentes hay a la vez
    async with semaforo:
        try:
            return await asyncio.to_thread(leer_y_analizar_csv, ruta)
        except Exception as e:
            logging.error(f"Error procesando {ruta}: {e}")
            return {"archivo": os.path.basename(ruta), "error": str(e)}


# ================================================================
# c) Escritura concurrente de los resúmenes
# ================================================================
def escribir_resumen_individual(resultados_validos):
    ruta = "resumen_individual.csv"
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["archivo", "filas", "media", "desviacion"] + [f"grupo_{g}" for g in GRUPOS])
        for r in resultados_validos:
            fila = [r["archivo"], r["filas"], round(r["media"], 4), round(r["desviacion"], 4)]
            fila += [r["conteo_grupo"].get(g, 0) for g in GRUPOS]
            writer.writerow(fila)
    return ruta


def escribir_resumen_global(resultados_validos):
    ruta = "resumen_global.txt"
    total_filas = sum(r["filas"] for r in resultados_validos)
    media_global = (
        sum(r["media"] * r["filas"] for r in resultados_validos) / total_filas if total_filas else 0.0
    )

    # Desviación global combinando las variabilidades parciales (fórmula estándar de combinación)
    variabilidad_global = 0.0
    if total_filas > 1: # Evitamos división por cero si hay menos de 2 filas en total
        suma_var = sum(
            (r["filas"] - 1) * (r["desviacion"] ** 2) + r["filas"] * (r["media"] - media_global) ** 2
            for r in resultados_validos
        )
        variabilidad_global = suma_var / (total_filas - 1)
    desviacion_global = variabilidad_global ** 0.5

    conteo_global = {g: 0 for g in GRUPOS}
    for r in resultados_validos: # Acumulamos el conteo de cada grupo de todos los archivos procesados
        for g in GRUPOS:
            conteo_global[g] += r["conteo_grupo"].get(g, 0)

    with open(ruta, "w", encoding="utf-8") as f:
        f.write(f"Total de filas: {total_filas}\n")
        f.write(f"Media global: {media_global:.4f}\n")
        f.write(f"Desviacion estandar global: {desviacion_global:.4f}\n")
        f.write("Conteo acumulado por grupo:\n")
        for g in GRUPOS:
            f.write(f"  {g}: {conteo_global[g]}\n")
    return ruta


async def escribir_resumenes(resultados_validos):
    return await asyncio.gather(
        asyncio.to_thread(escribir_resumen_individual, resultados_validos),
        asyncio.to_thread(escribir_resumen_global, resultados_validos),
    )


# ================================================================
# Pipelines completos: concurrente (b+c+d+f) y secuencial (e)
# ================================================================
async def pipeline_concurrente(rutas_csv, max_concurrencia=4):
    semaforo = asyncio.Semaphore(max_concurrencia)
    resultados = await asyncio.gather(*[procesar_csv(r, semaforo) for r in rutas_csv])

    validos = [r for r in resultados if "error" not in r]
    con_error = [r for r in resultados if "error" in r]

    await escribir_resumenes(validos)
    return validos, con_error


def pipeline_secuencial(rutas_csv):
    validos = []
    for ruta in rutas_csv:
        try:
            validos.append(leer_y_analizar_csv(ruta))
        except Exception as e:
            logging.error(f"Error procesando {ruta}: {e}")
    return validos


# ================================================================
# Main: ejecución de todo el pipeline y comparación de tiempos
# ================================================================
"""Función principal que ejecuta todo el pipeline: generación de CSV, 
procesamiento concurrente y secuencial, y comparación de tiempos."""
async def main():
    # ---- a) Generación: paralela vs secuencial ----
    print("=== OBJETIVO = a) Generando CSV (multiprocessing.Pool) ===")
    t0 = time.perf_counter()
    resultados_gen = generar_ficheros_paralelo(N_FICHEROS, procesos=4)
    t_gen_paralelo = time.perf_counter() - t0

    rutas_csv = [r for r in resultados_gen if not r.startswith("ERROR")]
    for r in resultados_gen:
        if r.startswith("ERROR"):
            logging.error(r)

    print(f"Ficheros generados: {len(rutas_csv)}")
    print(f"Tiempo generacion paralela: {t_gen_paralelo:.2f} s\n")

    # ---- b) + c) + d) + f): pipeline concurrente ----
    print("=== OBJETIVOS = b)/c)/d)/f) Procesamiento + resumenes concurrentes (asyncio) ===")
    t0 = time.perf_counter()
    validos, con_error = await pipeline_concurrente(rutas_csv, max_concurrencia=4)
    t_proc_concurrente = time.perf_counter() - t0

    print(f"Ficheros OK: {len(validos)} | Ficheros con error: {len(con_error)}")
    print(f"Tiempo procesamiento concurrente: {t_proc_concurrente:.2f} s\n")

    # ---- e) Comparacion con version secuencial ----
    print("=== OBJETIVO = e) Version secuencial (para comparar) ===")
    t0 = time.perf_counter()
    _ = pipeline_secuencial(rutas_csv)
    t_proc_secuencial = time.perf_counter() - t0
    print(f"Tiempo procesamiento secuencial: {t_proc_secuencial:.2f} s\n")

    print("=== Resumen de tiempos ===")
    print(f"Generacion paralela (Pool):        {t_gen_paralelo:.2f} s")
    print(f"Procesamiento concurrente (async): {t_proc_concurrente:.2f} s")
    print(f"Procesamiento secuencial:          {t_proc_secuencial:.2f} s")

# Ejecuta la función principal usando asyncio.run() si el script se ejecuta directamente
if __name__ == "__main__":
    asyncio.run(main())